
# B2. So sánh BM25 tự cài đặt với Pyserini

Mục tiêu: So sánh chất lượng truy xuất giữa BM25 tự cài đặt và Pyserini BM25 trên hai bộ dữ liệu SciFact và NFCorpus.

**Cấu hình thực nghiệm:**
- k1 = 1.2
- b = 0.75
- Top 100 tài liệu
- Chỉ số so sánh: nDCG@10


In [30]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env", override=True)
java_home = os.environ["JAVA_HOME"]
os.environ["PATH"] = java_home + r"\bin;" + os.environ["PATH"]

In [31]:
import subprocess

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

ket_qua = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(ket_qua.stderr)

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-21.0.13.9-hotspot
openjdk version "21.0.13" 2026-10-20 LTS
OpenJDK Runtime Environment Temurin-21.0.13+9 (build 21.0.13+9-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.13+9 (build 21.0.13+9-LTS, mixed mode, sharing)




### 3. Xây dựng chỉ mục Lucene

Pyserini sử dụng Apache Lucene để lập chỉ mục và truy xuất tài liệu.

Trong bước này, dữ liệu SciFact và NFCorpus được chuyển sang định dạng JSON phù hợp với Pyserini, sau đó xây dựng chỉ mục Lucene.

Các bộ dữ liệu được giữ nguyên nội dung tiêu đề và văn bản để phục vụ so sánh với BM25 tự cài đặt.


In [32]:

PHAN_B_DIR = Path.cwd().resolve()
DATA_DIR = PHAN_B_DIR / "du_lieu"

SCIFACT_DIR = DATA_DIR / "scifact"
NFCORPUS_DIR = DATA_DIR / "nfcorpus"

print("SciFact:", SCIFACT_DIR.exists())
print("NFCorpus:", NFCORPUS_DIR.exists())

SciFact: True
NFCorpus: True


# PYSERINI TRÊN SCIFACT


### 3.1. Chuẩn bị dữ liệu SciFact cho Pyserini

Đọc bộ dữ liệu SciFact bằng `GenericDataLoader` của BEIR, sau đó chuyển các tài liệu sang định dạng JSONL để xây dựng chỉ mục Lucene.

Mỗi dòng JSONL chứa:
- `id`: Mã tài liệu.
- `contents`: Nội dung tài liệu, được ghép từ tiêu đề và văn bản.

Cách ghép nội dung được giữ giống với BM25 tự cài đặt ở B1.


In [33]:
import json
from beir.datasets.data_loader import GenericDataLoader

scifact_corpus, scifact_queries, scifact_qrels = GenericDataLoader(
    data_folder=str(SCIFACT_DIR)
).load(split="test")
scifact_output_dir = DATA_DIR / "pyserini" / "scifact"
scifact_output_dir.mkdir(parents=True, exist_ok=True)

scifact_output_file = scifact_output_dir / "docs.jsonl"

with open(scifact_output_file, "w", encoding="utf-8") as f:
    for doc_id, doc in scifact_corpus.items():
        record = {
            "id": doc_id,
            "contents": doc["title"] + " " + doc["text"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        
print("Số tài liệu:", len(scifact_corpus))
print("File JSONL:", scifact_output_file)

100%|██████████| 5183/5183 [00:00<00:00, 161950.32it/s]

Số tài liệu: 5183
File JSONL: D:\RAG\tep_kem\phanB\du_lieu\pyserini\scifact\docs.jsonl


In [34]:
with open(scifact_output_file, "r", encoding="utf-8") as f:
    for _ in range(2):
        print(f.readline()[:300])

{"id": "4983", "contents": "Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functio
{"id": "5836", "contents": "Induction of myelodysplasia by myeloid-derived suppressor cells. Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share biological features of activated adaptive immune response and ineffective hematopoiesis. Here we report that myeloid-derive



### 3.2. Xây dựng chỉ mục Lucene cho SciFact

Sử dụng Pyserini để xây dựng chỉ mục Lucene từ 5.183 tài liệu SciFact đã chuyển sang JSONL.

Chỉ mục lưu thông tin cần thiết để truy xuất BM25. Mã tài liệu được giữ nguyên nhằm đối chiếu kết quả truy xuất với qrels.

Sử dụng chế độ `JsonCollection` và lưu vị trí tài liệu trong chỉ mục để phục vụ tìm kiếm.


In [35]:
import subprocess
import sys

scifact_index_dir = DATA_DIR / "pyserini" / "indexes" / "scifact"

if scifact_index_dir.exists() and any(scifact_index_dir.glob("segments_*")):
    print("Index SciFact đã tồn tại, bỏ qua tạo mới.")
else:
    lenh = [
        sys.executable,
        "-m", "pyserini.index.lucene",
        "--collection", "JsonCollection",
        "--input", str(scifact_output_dir),
        "--index", str(scifact_index_dir),
        "--generator", "DefaultLuceneDocumentGenerator",
        "--threads", "1",
        "--storePositions",
        "--storeDocvectors",
        "--storeRaw"
    ]

    ket_qua = subprocess.run(
        lenh,
        capture_output=True,
        text=True
    )

    print("Return code:", ket_qua.returncode)
    print("STDOUT:", ket_qua.stdout)
    print("STDERR:", ket_qua.stderr)

    if ket_qua.returncode != 0:
        raise RuntimeError("Tạo index SciFact thất bại")

Index SciFact đã tồn tại, bỏ qua tạo mới.


In [36]:
from pyserini.index.lucene import LuceneIndexReader

scifact_reader = LuceneIndexReader(str(scifact_index_dir))

print("Thống kê chỉ mục:", scifact_reader.stats())

Thống kê chỉ mục: {'total_terms': 838128, 'documents': 5183, 'non_empty_documents': 5183, 'unique_terms': 28865}


**Nhận xét thống kê chỉ mục SciFact:**

- `total_terms = 838128`: Tổng số lượt xuất hiện của các từ được lập chỉ mục sau tiền xử lý.
- `documents = 5183`: Tổng số tài liệu đã được lập chỉ mục.
- `non_empty_documents = 5183`: Cả 5.183 tài liệu đều có nội dung được lập chỉ mục, không có tài liệu rỗng.
- `unique_terms = 28865`: Số lượng từ khác nhau trong chỉ mục sau tiền xử lý.

**Kết luận:** Pyserini đã lập chỉ mục thành công toàn bộ 5.183 tài liệu của SciFact. Chỉ mục chứa 28.865 từ phân biệt, phục vụ việc truy xuất và xếp hạng tài liệu bằng BM25.

### 3.5. Truy xuất BM25 bằng Pyserini


Sử dụng `LuceneSearcher` để truy xuất tài liệu từ chỉ mục NFCorpus.

Cấu hình:
- `k1 = 1.2`, `b = 0.75`: Tham số BM25 giống với phương pháp tự cài đặt.
- `k = 100`: Lấy 100 tài liệu có điểm BM25 cao nhất cho mỗi truy vấn.

Lưu kết quả dưới dạng dictionary, trong đó mỗi `query_id` tương ứng với danh sách `(doc_id, score)` được sắp xếp theo điểm BM25 giảm dần.

In [37]:
from pyserini.search.lucene import LuceneSearcher

scifact_searcher = LuceneSearcher(str(scifact_index_dir))
scifact_searcher.set_bm25(k1=1.2, b=0.75)

query = next(iter(scifact_queries.values()))

ket_qua = scifact_searcher.search(query, k=5) #diem

print("Truy vấn mẫu:", query)
print("Top 3 tài liệu:")

for hit in ket_qua[:3]:
    print(hit.docid, hit.score)

Truy vấn mẫu: 0-dimensional biomaterials show inductive properties.
Top 3 tài liệu:
10608397 6.063499927520752
18953920 4.555500030517578
34386619 4.492400169372559


### 3.6. Truy xuất toàn bộ truy vấn SciFact

Sử dụng Pyserini BM25 với k1 = 1.2, b = 0.75 để truy xuất top 100 tài liệu cho mỗi truy vấn trong tập test SciFact.

In [38]:
ket_qua_scifact_pyserini = {}

for query_id, query_text in scifact_queries.items():
    hits = scifact_searcher.search(query_text, k=100)

    ket_qua_scifact_pyserini[query_id] = [
        (hit.docid, hit.score)
        for hit in hits
    ]

print("Số truy vấn:", len(ket_qua_scifact_pyserini))
for query_id, ket_qua in list(ket_qua_scifact_pyserini.items())[:3]:
    print({query_id: ket_qua[:3]})
print("Số tài liệu truy xuất của truy vấn đầu:", len(next(iter(ket_qua_scifact_pyserini.values()))))

Số truy vấn: 300
{'1': [('10608397', 6.063499927520752), ('18953920', 4.555500030517578), ('34386619', 4.492400169372559)]}
{'3': [('2739854', 15.932299613952637), ('14717500', 15.077400207519531), ('3672261', 14.82349967956543)]}
{'5': [('13734012', 10.417900085449219), ('18617259', 8.555999755859375), ('17333231', 5.682700157165527)]}
Số tài liệu truy xuất của truy vấn đầu: 100


### 3.7. Đánh giá nDCG@10 trên SciFact

Sử dụng hàm `ndcg_at_k()` đã xây dựng để đánh giá chất lượng xếp hạng tài liệu của Pyserini BM25.

Với mỗi truy vấn:
- Lấy danh sách `doc_id` từ kết quả truy xuất.
- Đối chiếu với nhãn liên quan trong `scifact_qrels`.
- Tính nDCG@10 cho 10 tài liệu đứng đầu.

Kết quả cuối cùng là giá trị nDCG@10 trung bình trên 300 truy vấn của tập test SciFact.

In [39]:
import sys

sys.path.insert(0, str(PHAN_B_DIR.parent))
from chi_so import ndcg_at_k

ndcg_scifact_pyserini = sum(
    ndcg_at_k(
        [doc_id for doc_id, score in ket_qua_scifact_pyserini[query_id]],
        scifact_qrels.get(query_id, {}),
        10
    )
    for query_id in scifact_queries
) / len(scifact_queries)

print(f"nDCG@10 Pyserini (SciFact): {ndcg_scifact_pyserini:.4f}")

nDCG@10 Pyserini (SciFact): 0.6826


**Nhận xét:** Pyserini BM25 đạt nDCG@10 = **0.6826** trên 300 truy vấn của bộ dữ liệu SciFact, cao hơn BM25 tự cài đặt (0.5723) khoảng **0.1103 điểm**. Kết quả cho thấy Pyserini xếp hạng các tài liệu liên quan trong top 10 tốt hơn. Sự chênh lệch có thể đến từ cách tiền xử lý văn bản, đặc biệt là việc Pyserini sử dụng Lucene Analyzer với stopword removal và stemming, trong khi BM25 tự cài đặt sử dụng tách từ theo khoảng trắng.

# 4. Pyserini trên NFCorpus

### 4.1. Chuẩn bị dữ liệu NFCorpus cho Pyserini

Sử dụng `GenericDataLoader` của BEIR để nạp dữ liệu NFCorpus.

Chuyển các tài liệu sang định dạng JSONL gồm hai trường:
- `id`: Mã tài liệu.
- `contents`: Tiêu đề ghép với nội dung văn bản.

Cách ghép nội dung được giữ giống BM25 tự cài đặt ở B1.

In [40]:
nfcorpus_corpus, nfcorpus_queries, nfcorpus_qrels = GenericDataLoader(
    data_folder=str(NFCORPUS_DIR)
).load(split="test")

nfcorpus_output_dir = DATA_DIR / "pyserini" / "nfcorpus"
nfcorpus_output_dir.mkdir(parents=True, exist_ok=True)

nfcorpus_output_file = nfcorpus_output_dir / "docs.jsonl"

with open(nfcorpus_output_file, "w", encoding="utf-8") as f:
    for doc_id, doc in nfcorpus_corpus.items():
        record = {
            "id": doc_id,
            "contents": doc["title"] + " " + doc["text"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Số tài liệu:", len(nfcorpus_corpus))
print("File JSONL:", nfcorpus_output_file)

100%|██████████| 3633/3633 [00:00<00:00, 187582.71it/s]

Số tài liệu: 3633
File JSONL: D:\RAG\tep_kem\phanB\du_lieu\pyserini\nfcorpus\docs.jsonl


Kiểm tra hai tài liệu đầu tiên trong file JSONL sau khi chuyển đổi.

In [41]:
with open(nfcorpus_output_file, "r", encoding="utf-8") as f:
    for _ in range(2):
        print(f.readline()[:300])

{"id": "MED-10", "contents": "Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-s
{"id": "MED-14", "contents": "Statin use after diagnosis of breast cancer and survival: a population-based cohort study. BACKGROUND: Preclinical studies have shown that statins, particularly simvastatin, can prevent growth in breast cancer cell lines and animal models. We investigated whether statin


### 4.3. Tạo chỉ mục Lucene cho NFCorpus

Sử dụng Pyserini để xây dựng chỉ mục Lucene từ file `docs.jsonl` đã chuẩn bị.

Cấu hình:
- `JsonCollection`: Đọc tài liệu từ định dạng JSONL.
- `DefaultLuceneDocumentGenerator`: Chuyển tài liệu thành định dạng Lucene.
- `--threads 1`: Sử dụng một luồng lập chỉ mục.
- `--storePositions`, `--storeDocvectors`, `--storeRaw`: Lưu thêm thông tin phục vụ truy xuất và kiểm tra.

Nếu chỉ mục đã tồn tại thì bỏ qua bước tạo mới để tránh sử dụng thêm dung lượng ổ đĩa.

In [42]:
nfcorpus_index_dir = DATA_DIR / "pyserini" / "indexes" / "nfcorpus"

if nfcorpus_index_dir.exists() and any(nfcorpus_index_dir.glob("segments_*")):
    print("Index NFCorpus đã tồn tại, bỏ qua tạo mới.")
else:
    lenh = [
        sys.executable,
        "-m", "pyserini.index.lucene",
        "--collection", "JsonCollection",
        "--input", str(nfcorpus_output_dir),
        "--index", str(nfcorpus_index_dir),
        "--generator", "DefaultLuceneDocumentGenerator",
        "--threads", "1",
        "--storePositions",
        "--storeDocvectors",
        "--storeRaw"
    ]

    ket_qua = subprocess.run(
        lenh,
        capture_output=True,
        text=True
    )

    print("Return code:", ket_qua.returncode)
    print("STDOUT:", ket_qua.stdout)
    print("STDERR:", ket_qua.stderr)

    if ket_qua.returncode != 0:
        raise RuntimeError("Tạo index NFCorpus thất bại")

Index NFCorpus đã tồn tại, bỏ qua tạo mới.


### 4.4. Kiểm tra và thống kê chỉ mục NFCorpus

Sử dụng `LuceneIndexReader` để đọc chỉ mục đã tạo và thống kê số tài liệu, tổng số từ và số từ phân biệt.

In [43]:
nfcorpus_reader = LuceneIndexReader(str(nfcorpus_index_dir))

print("Thống kê chỉ mục NFCorpus:", nfcorpus_reader.stats())

Thống kê chỉ mục NFCorpus: {'total_terms': 637485, 'documents': 3633, 'non_empty_documents': 3633, 'unique_terms': 22111}


**Nhận xét thống kê chỉ mục NFCorpus:**

- `total_terms = 637485`: Tổng số lượt xuất hiện của các từ được lập chỉ mục sau tiền xử lý.
- `documents = 3633`: Tổng số tài liệu đã được lập chỉ mục.
- `non_empty_documents = 3633`: Cả 3.633 tài liệu đều có nội dung được lập chỉ mục, không có tài liệu rỗng.
- `unique_terms = 22111`: Số lượng từ khác nhau trong chỉ mục sau tiền xử lý.

**Kết luận:** Pyserini đã lập chỉ mục thành công toàn bộ 3.633 tài liệu của NFCorpus. Chỉ mục chứa 22.111 từ phân biệt, phục vụ việc truy xuất và xếp hạng tài liệu bằng BM25.

### 4.5. Truy xuất tài liệu bằng Pyserini BM25

Sử dụng `LuceneSearcher` để truy xuất tài liệu từ chỉ mục NFCorpus.

Cấu hình:
- `k1 = 1.2`, `b = 0.75`: Tham số BM25 giống với phương pháp tự cài đặt.
- `k = 100`: Lấy 100 tài liệu có điểm BM25 cao nhất cho mỗi truy vấn.

Lưu kết quả dưới dạng dictionary, trong đó mỗi `query_id` tương ứng với danh sách `(doc_id, score)` được sắp xếp theo điểm BM25 giảm dần.

In [46]:
nfcorpus_searcher = LuceneSearcher(str(nfcorpus_index_dir))
nfcorpus_searcher.set_bm25(k1=1.2, b=0.75)

query = next(iter(nfcorpus_queries.values()))

ket_qua = nfcorpus_searcher.search(query, k=5) 


print("Truy vấn mẫu:", query)
print("Top 3 tài liệu:")

for hit in ket_qua[:3]:
    print(hit.docid, hit.score)    

Truy vấn mẫu: Do Cholesterol Statin Drugs Cause Breast Cancer?
Top 3 tài liệu:
MED-10 9.091899871826172
MED-14 9.001899719238281
MED-2429 8.9552001953125


### 4.6. Truy xuất toàn bộ truy vấn NFCorpus

Sử dụng Pyserini BM25 với k1 = 1.2, b = 0.75 để truy xuất top 100 tài liệu cho mỗi truy vấn trong tập test NFCorpus.

In [47]:
ket_qua_nfcorpus_pyserini = {}

for query_id, query_text in nfcorpus_queries.items():
    hits = nfcorpus_searcher.search(query_text, k=100)

    ket_qua_nfcorpus_pyserini[query_id] = [
        (hit.docid, hit.score)
        for hit in hits
    ]

print("Số truy vấn:", len(ket_qua_nfcorpus_pyserini))

for query_id, ket_qua in list(ket_qua_nfcorpus_pyserini.items())[:3]:
    print({query_id: ket_qua[:3]})

print(
    "Số tài liệu truy xuất của truy vấn đầu:",
    len(next(iter(ket_qua_nfcorpus_pyserini.values())))
)

Số truy vấn: 323
{'PLAIN-2': [('MED-10', 9.091899871826172), ('MED-14', 9.001899719238281), ('MED-2429', 8.9552001953125)]}
{'PLAIN-12': [('MED-4711', 5.310400009155273), ('MED-2501', 4.776800155639648), ('MED-4262', 4.322999954223633)]}
{'PLAIN-23': [('MED-2643', 5.3379998207092285), ('MED-2661', 5.203199863433838), ('MED-2652', 5.152100086212158)]}
Số tài liệu truy xuất của truy vấn đầu: 100


### 4.7. Đánh giá nDCG@10 trên NFCorpus

Sử dụng hàm `ndcg_at_k()` đã xây dựng để đánh giá chất lượng xếp hạng tài liệu của Pyserini BM25.

Với mỗi truy vấn:
- Lấy danh sách `doc_id` từ kết quả truy xuất.
- Đối chiếu với nhãn liên quan trong `nfcorpus_qrels`.
- Tính nDCG@10 cho 10 tài liệu đứng đầu.

Kết quả cuối cùng là giá trị nDCG@10 trung bình trên 323 truy vấn của tập test NFCorpus.

In [49]:
ndcg_nfcorpus_pyserini = sum(
    ndcg_at_k(
        [doc_id for doc_id, score in ket_qua_nfcorpus_pyserini[query_id]],
        nfcorpus_qrels.get(query_id, {}),
        10
    )
    for query_id in nfcorpus_queries
) / len(nfcorpus_queries)

print(f"nDCG@10 Pyserini (NFCorpus): {ndcg_nfcorpus_pyserini:.4f}")

nDCG@10 Pyserini (NFCorpus): 0.3222


**Nhận xét kết quả đánh giá NFCorpus:**

- Pyserini BM25 đạt **nDCG@10 = 0.3222** trên 323 truy vấn của tập test NFCorpus.
- BM25 tự cài đặt đạt **nDCG@10 = 0.2697**.
- Pyserini có kết quả cao hơn **0.0525**, cho thấy chất lượng xếp hạng 10 tài liệu đầu tiên tốt hơn theo thước đo nDCG@10.

**Kết luận:** Pyserini BM25 cho kết quả truy xuất tốt hơn BM25 tự cài đặt trên NFCorpus. Sự khác biệt có thể đến từ cách tiền xử lý văn bản, phân tích từ vựng và các chi tiết triển khai BM25 của Lucene.

## 5. So sánh BM25 tự cài đặt và Pyserini BM25

### 5.1. Bảng kết quả nDCG@10

| Bộ dữ liệu | BM25 tự cài đặt | Pyserini BM25 | Chênh lệch |
|---|---:|---:|---:|
| SciFact | 0.5723 | 0.6826 | +0.1103 |
| NFCorpus | 0.2697 | 0.3222 | +0.0525 |

### 5.2. Nhận xét

- **SciFact:** Pyserini đạt nDCG@10 cao hơn BM25 tự cài đặt 0.1103.
- **NFCorpus:** Pyserini đạt nDCG@10 cao hơn BM25 tự cài đặt 0.0525.
- Trên cả hai bộ dữ liệu, Pyserini BM25 đều có chất lượng xếp hạng tốt hơn theo chỉ số nDCG@10.
- Một nguyên nhân có thể là sự khác biệt trong tiền xử lý văn bản, bao gồm tách từ, loại bỏ stopwords và stemming.
- Kết quả cũng cho thấy việc sử dụng cùng tham số `k1 = 1.2` và `b = 0.75` không đảm bảo hai hệ thống cho kết quả giống nhau.

### 5.3. Kết luận

Thực nghiệm cho thấy Pyserini BM25 đạt hiệu quả truy xuất cao hơn phiên bản BM25 tự cài đặt trên cả SciFact và NFCorpus. Điều này cho thấy cách xử lý văn bản và triển khai hệ thống truy xuất có thể ảnh hưởng đáng kể đến chất lượng xếp hạng tài liệu.